# 🗨️ 02 · 视觉语言模型：BLIP / Qwen-VL / LLaVA

> 从「CLIP 会算相似度」到「模型会看图说话」，中间是**视觉语言模型（VLM）**。
> 本节讲清两条主流路线——**Q-Former 对齐（BLIP-2）** 与 **投影层+指令微调（LLaVA）**。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | VLM 怎么把图像接进 LLM？ | 视觉 token → 对齐层 → 语言模型 |
| 2 | BLIP-2 怎么对齐？ | Q-Former（可学习 query） |
| 3 | LLaVA 怎么对齐？ | 线性投影 + 指令微调 |
| 4 | 两类方法怎么选？ | 冻结/微调、能力平衡 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ VLM 总体架构（必背）

In [2]:
print('''通用 VLM 架构：
  图像 -> 视觉编码器(ViT) -> 视觉 token(如 256/576 个)
       -> 对齐层(Projector/Q-Former/MLP)
       -> 语言模型(LLM) -> 文本输出

两条主流路线：
  A. Q-Former(BLIP-2)：可学习 query 与视觉特征交叉注意力，压缩视觉信息；
  B. 线性/MLP 投影(LLaVA)：视觉 token 直接投影进 LLM 的 embedding 空间；
  C. 原生多模态(最新)：LLM 内部直接处理多模态 token（如 Qwen-VL、Gemini 的做法）。
''')

通用 VLM 架构：
  图像 -> 视觉编码器(ViT) -> 视觉 token(如 256/576 个)
       -> 对齐层(Projector/Q-Former/MLP)
       -> 语言模型(LLM) -> 文本输出

两条主流路线：
  A. Q-Former(BLIP-2)：可学习 query 与视觉特征交叉注意力，压缩视觉信息；
  B. 线性/MLP 投影(LLaVA)：视觉 token 直接投影进 LLM 的 embedding 空间；
  C. 原生多模态(最新)：LLM 内部直接处理多模态 token（如 Qwen-VL、Gemini 的做法）。



## 2️⃣ 手写"视觉 token → 对齐 → LLM" 流程

In [3]:
# 模拟：视觉编码器输出 256 个 patch 特征；LLM 输入空间 4096 维
n_visual = 256
vis_feat = rng.normal(0, 1, (n_visual, 1024))      # 视觉特征（如 ViT 输出）

# 路线 A：Q-Former 压缩到 32 个 query（简化：query 与视觉特征做注意力池化）
n_query = 32
Q = rng.normal(0, 0.1, (n_query, 1024))            # 可学习 query
scores = Q @ vis_feat.T / np.sqrt(1024)
attn_w = np.exp(scores - scores.max(axis=1, keepdims=True))
attn_w = attn_w / attn_w.sum(axis=1, keepdims=True)
qformer_out = attn_w @ vis_feat                     # (32, 1024)
print(f'Q-Former: {n_visual} 个视觉 token -> {n_query} 个 query（信息压缩）')

# 路线 B：线性投影直接把视觉 token 投到 LLM 空间
proj = rng.normal(0, 0.1, (1024, 4096))
llm_input_proj = vis_feat @ proj                    # (256, 4096)
print(f'线性投影: (256, 1024) -> {llm_input_proj.shape}')
assert qformer_out.shape == (n_query, 1024) and llm_input_proj.shape == (256, 4096)
print('✅ 两条对齐路线都能把视觉信息接入 LLM')

Q-Former: 256 个视觉 token -> 32 个 query（信息压缩）


线性投影: (256, 1024) -> (256, 4096)
✅ 两条对齐路线都能把视觉信息接入 LLM


## 3️⃣ 对齐训练三阶段（BLIP-2/LLaVA 风格）

In [4]:
print('''三阶段训练范式：
  阶段1：冻结视觉编码器+LLM，只训对齐层（图像-文本对比/生成对齐）；
  阶段2：冻结视觉编码器+对齐层，微调 LLM（指令微调，学会看图回答）；
  阶段3（可选）：全量微调 / 视觉语言联合 SFT / RLHF。

面试点：
  为什么冻结？→ 省显存、防灾难性遗忘、对齐层先学会"翻译"；
  视觉 token 太多怎么办？→ Q-Former 压缩 / 平均池化 / 视觉 token 降采样；
  训练数据？→ 图文对(预训练) + 指令数据(微调,如 LLaVA-Instruct)。
''')

三阶段训练范式：
  阶段1：冻结视觉编码器+LLM，只训对齐层（图像-文本对比/生成对齐）；
  阶段2：冻结视觉编码器+对齐层，微调 LLM（指令微调，学会看图回答）；
  阶段3（可选）：全量微调 / 视觉语言联合 SFT / RLHF。

面试点：
  为什么冻结？→ 省显存、防灾难性遗忘、对齐层先学会"翻译"；
  视觉 token 太多怎么办？→ Q-Former 压缩 / 平均池化 / 视觉 token 降采样；
  训练数据？→ 图文对(预训练) + 指令数据(微调,如 LLaVA-Instruct)。



## 4️⃣ 图文对话推理流程（示意）

In [5]:
print('''推理流程（以 LLaVA 为例）：
  1. 用户上传图片
  2. ViT 编码 -> 576 个视觉 token
  3. MLP 投影 -> LLM embedding 空间
  4. 拼接文本指令 token（如 "请描述这张图"）
  5. LLM 自回归生成回答

示例：
  图片：一只橘猫在窗台晒太阳
  指令：Describe the image.
  回答：A ginger cat is lying on a windowsill, basking in the sunlight.
''')
sample_visual = rng.normal(0, 1, (5, 1024))  # 5 个示例 token（象征性）
print('视觉 token 示例 (5, 1024):', sample_visual.shape)

推理流程（以 LLaVA 为例）：
  1. 用户上传图片
  2. ViT 编码 -> 576 个视觉 token
  3. MLP 投影 -> LLM embedding 空间
  4. 拼接文本指令 token（如 "请描述这张图"）
  5. LLM 自回归生成回答

示例：
  图片：一只橘猫在窗台晒太阳
  指令：Describe the image.
  回答：A ginger cat is lying on a windowsill, basking in the sunlight.

视觉 token 示例 (5, 1024): (5, 1024)


## 5️⃣ BLIP vs LLaVA（面试对比）

In [6]:
print('''| 维度 | BLIP-2 | LLaVA |
|---|---|---|
| 对齐方式 | Q-Former（交叉注意力压缩） | 简单 MLP/线性投影 |
| 视觉信息 | 压缩到固定 query 数 | 保留全部 token（可加池化） |
| 参数量 | 中 | 轻 |
| 优点 | 对齐更充分、信息压缩可控 | 简单、微调成本低、社区迭代快 |
| 缺点 | 结构复杂、训练分阶段 | 视觉细节保留欠佳（早期版本） |
| 代表 | BLIP-2 / InstructBLIP | LLaVA-1.5/1.6 |

演进趋势：Qwen-VL/Gemini/Llama-3.2-Vision 走向"统一原生多模态"。
''')

| 维度 | BLIP-2 | LLaVA |
|---|---|---|
| 对齐方式 | Q-Former（交叉注意力压缩） | 简单 MLP/线性投影 |
| 视觉信息 | 压缩到固定 query 数 | 保留全部 token（可加池化） |
| 参数量 | 中 | 轻 |
| 优点 | 对齐更充分、信息压缩可控 | 简单、微调成本低、社区迭代快 |
| 缺点 | 结构复杂、训练分阶段 | 视觉细节保留欠佳（早期版本） |
| 代表 | BLIP-2 / InstructBLIP | LLaVA-1.5/1.6 |

演进趋势：Qwen-VL/Gemini/Llama-3.2-Vision 走向"统一原生多模态"。



## 6️⃣ 视觉 token 数量的权衡（高频追问）

In [7]:
# 视觉 token 数直接决定 LLM 输入长度与显存
for n in [64, 256, 576, 1024]:
    print(f'视觉 token {n} 个：序列长度=文本+n，注意力 O((T+n)^2)')
print('''
  少(token)：推理快、显存省，但细粒度信息丢失（认不清小物体/文字）；
  多(token)：细节好，但计算贵、长上下文压力大；
  方案：分辨率分桶/降采样/视觉 token 压缩/只看 ROI 区域。
''')

视觉 token 64 个：序列长度=文本+n，注意力 O((T+n)^2)
视觉 token 256 个：序列长度=文本+n，注意力 O((T+n)^2)
视觉 token 576 个：序列长度=文本+n，注意力 O((T+n)^2)
视觉 token 1024 个：序列长度=文本+n，注意力 O((T+n)^2)

  少(token)：推理快、显存省，但细粒度信息丢失（认不清小物体/文字）；
  多(token)：细节好，但计算贵、长上下文压力大；
  方案：分辨率分桶/降采样/视觉 token 压缩/只看 ROI 区域。



## 📝 自测清单

- [ ] 能画出 VLM 通用架构（视觉编码器→对齐层→LLM）
- [ ] 能讲清 Q-Former 与线性投影两条对齐路线
- [ ] 能背出三阶段训练范式与冻结原因
- [ ] 能对比 BLIP-2 与 LLaVA 的优劣
- [ ] 能回答"视觉 token 太多/太少怎么办"追问